## Многоклассовая классификация с помощью RuBERT

Установим и импортируем нужные библиотеки:

[Transformers](https://huggingface.co/docs/transformers/index) - популярная библиотека, в которой реализованы архитектуры нейронных сетей для обработки текста, изображений, аудио а также мультимодальные архитектуры.

In [2]:
!pip install transformers

In [3]:
import pandas as pd
import numpy as np
from sklearn import metrics
import transformers
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, BertConfig
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm import tqdm

In [4]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

**Важно!**

Уважаемые участники! Для воспроизводимости ваших результатов при проверке, обязательно фиксируйте seed генераторов случайных чисел при обучении ваших алгоритмов. Записывайте seed в имя файла с ответами. Выбирайте seed каждый раз случайным образом и всегда записывайте его в имя файла с ответами для воспроизводимости результатов.

In [5]:
import random

# Если есть желание сравнивать модели между собой, то seed лучше устанаваливать всегда одним и тем же
# seed = random.randint(1, 1000)
seed=931
print(f'seed: {seed}')

random.seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)

seed: 931


Прочитаем набор с данными (train.csv).

Если указанная команда не заработала, скопируйте архивы с данными на свой Google Drive и укажите id архивов из ссылок к вашему Google Drive.

In [6]:
!gdown 1JBldDLK2wrgfD1CheAv4FKsLBf71Avis

Downloading...
From: https://drive.google.com/uc?id=1JBldDLK2wrgfD1CheAv4FKsLBf71Avis
To: /content/train.csv
100% 3.67M/3.67M [00:00<00:00, 160MB/s]


In [7]:
df = pd.read_csv('train.csv')
df = df.rename({'Фильм': 'movie', 'Сюжет': 'plot', 'Жанры': 'genres', 'Описание': 'description'}, axis=1)
df

,movie,description,plot,genres
0,Дивергент (2014),"Действие фильма «Дивергент» происходит в мире,...","Действие фильма «Дивергент» происходит в мире,...","фантастика, детектив, боевик, мелодрама"
1,Кунг-фу Панда 4 (2024),Однажды ночью на вершине горы возле каменоломн...,Однажды ночью на вершине горы возле каменоломн...,"мультфильм, фэнтези, боевик, комедия, приключения"
2,2046 (2004),Чоу возвращается в Гонконг после нескольких ле...,Чоу возвращается в Гонконг после нескольких ле...,"фантастика, драма, мелодрама"
3,Полицейский из Беверли-Хиллз: Аксель Фоули (2024),Аксель Фоули вернулся в Беверли-Хиллз после то...,Аксель Фоули вернулся в Беверли-Хиллз после то...,"боевик, комедия, криминал, детектив"
4,"Знакомьтесь, Джо Блэк (1998)","История об Ангеле Смерти, который решает взять...","История об Ангеле Смерти, который решает взять...","мелодрама, фэнтези, драма"
...,...,...,...,...
566,Апокалипсис (2006),В 1517 году на полуострове Юкатан племя Лапы Я...,1517 год. Полуостров Юкатан. Группа охотников ...,"боевик, триллер, драма, приключения"
567,Лёд 3 (2024),"Надя, ставшая фигуристкой, стремится выиграть ...",Фильм начинается с истории взросления дочери А...,"мюзикл, мелодрама"
568,Дастур (2023),"Новоиспеченная невеста, которую выдали замуж п...",NaN,"ужасы, фантастика"
569,Не говори никому (2024),Пара вместе с дочерью получают приглашение от ...,Пара вместе с дочерью получают приглашение от ...,"триллер, драма"


In [ ]:
df.isnull().any()

movie          False
description    False
plot            True
genres         False
dtype: bool

Заполним пустые значения столбца "Сюжет" значениями из столбца "Описание"

In [8]:
df['plot'].fillna(df['description'], inplace=True)
df.isnull().any()

,0
movie,False
description,False
plot,False
genres,False


In [ ]:
df.shape

(571, 4)

Преобразуем столбец с жанрами к нужному формату (списку из 0 и 1):

In [9]:
df['genres'] = df['genres'].apply(lambda x: x.split(', '))

In [10]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['genres'])
df['target'] = y.tolist()
df

,movie,description,plot,genres,target
0,Дивергент (2014),"Действие фильма «Дивергент» происходит в мире,...","Действие фильма «Дивергент» происходит в мире,...","[фантастика, детектив, боевик, мелодрама]","[1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]"
1,Кунг-фу Панда 4 (2024),Однажды ночью на вершине горы возле каменоломн...,Однажды ночью на вершине горы возле каменоломн...,"[мультфильм, фэнтези, боевик, комедия, приключ...","[1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1]"
2,2046 (2004),Чоу возвращается в Гонконг после нескольких ле...,Чоу возвращается в Гонконг после нескольких ле...,"[фантастика, драма, мелодрама]","[0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]"
3,Полицейский из Беверли-Хиллз: Аксель Фоули (2024),Аксель Фоули вернулся в Беверли-Хиллз после то...,Аксель Фоули вернулся в Беверли-Хиллз после то...,"[боевик, комедия, криминал, детектив]","[1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]"
4,"Знакомьтесь, Джо Блэк (1998)","История об Ангеле Смерти, который решает взять...","История об Ангеле Смерти, который решает взять...","[мелодрама, фэнтези, драма]","[0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]"
...,...,...,...,...,...
566,Апокалипсис (2006),В 1517 году на полуострове Юкатан племя Лапы Я...,1517 год. Полуостров Юкатан. Группа охотников ...,"[боевик, триллер, драма, приключения]","[1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0]"
567,Лёд 3 (2024),"Надя, ставшая фигуристкой, стремится выиграть ...",Фильм начинается с истории взросления дочери А...,"[мюзикл, мелодрама]","[0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]"
568,Дастур (2023),"Новоиспеченная невеста, которую выдали замуж п...","Новоиспеченная невеста, которую выдали замуж п...","[ужасы, фантастика]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0]"
569,Не говори никому (2024),Пара вместе с дочерью получают приглашение от ...,Пара вместе с дочерью получают приглашение от ...,"[триллер, драма]","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]"


Порядок присвоенных классов можно посмотреть так:

In [11]:
mlb.classes_

array(['боевик', 'детектив', 'драма', 'история', 'комедия', 'криминал',
       'мелодрама', 'мультфильм', 'мюзикл', 'приключения', 'триллер',
       'ужасы', 'фантастика', 'фэнтези'], dtype=object)

Будем выполнять классификацию только на основе сюжета. Оставим сюжет и разметку.

In [12]:
df = df[['plot', 'target']].copy()
df

,plot,target
0,"Действие фильма «Дивергент» происходит в мире,...","[1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]"
1,Однажды ночью на вершине горы возле каменоломн...,"[1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1]"
2,Чоу возвращается в Гонконг после нескольких ле...,"[0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0]"
3,Аксель Фоули вернулся в Беверли-Хиллз после то...,"[1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]"
4,"История об Ангеле Смерти, который решает взять...","[0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]"
...,...,...
566,1517 год. Полуостров Юкатан. Группа охотников ...,"[1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0]"
567,Фильм начинается с истории взросления дочери А...,"[0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0]"
568,"Новоиспеченная невеста, которую выдали замуж п...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0]"
569,Пара вместе с дочерью получают приглашение от ...,"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]"


Конфигурация сети и подготовка набора.

MAX_LEN - максимальная длина последовательности токенов, которую будет обрабатывать модель. Модель BERT может обрабатывать до 512 токенов. Токены - слова, обработанные токенизатором. Будем использовать BertTokenizer.

In [13]:
MAX_LEN = 512
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 5e-05
tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Создадим класс-датасет для получения данных в нужном формате для сети BERT.

Мы токенизируем данные и возвращаем полученные с помощью токенизатора значения ids (индексы полученных токенов), mask (маска механизма attention (внимания) -- показывает, какие токены важные, а какие нет), token_type_ids (нужны для того, чтобы модель понимала контекст - где начинается и заканчивается последовательность).

In [14]:
class TrainDataset(Dataset):

  def __init__(self, dataframe, tokenizer, max_len):
    self.tokenizer = tokenizer
    self.data = dataframe
    self.text = self.data['plot']
    self.targets = self.data.target
    self.max_len = max_len

  def __len__(self):
    return len(self.text)

  def __getitem__(self, index):
    text = str(self.text[index])

    inputs = self.tokenizer.encode_plus(
        text,
        None,
        add_special_tokens=True,
        max_length=self.max_len,
        padding='max_length',
        return_token_type_ids=True,
        truncation=True
    )
    ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    token_type_ids = inputs["token_type_ids"]

    return {
        'ids': torch.tensor(ids, dtype=torch.long),
        'mask': torch.tensor(mask, dtype=torch.long),
        'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
        'targets': torch.tensor(self.targets[index], dtype=torch.float)
    }

In [15]:
# Разделяем DataFrame на train и val часть
train_size = 0.8
train_dataset = df.sample(frac=train_size, random_state=seed)
val_dataset = df.drop(train_dataset.index).reset_index(drop=True)
train_dataset.reset_index(drop=True, inplace=True)

print(f"FULL dataset: {df.shape}")
print(f"TRAIN dataset: {train_dataset.shape}")
print(f"VAL dataset: {val_dataset.shape}")

# Создаем наборы объектов
training_set = TrainDataset(train_dataset, tokenizer, MAX_LEN)
validation_set = TrainDataset(val_dataset, tokenizer, MAX_LEN)

FULL dataset: (571, 2)
TRAIN dataset: (457, 2)
VAL dataset: (114, 2)


In [16]:
# Создаем DataLoader для каждого из наборов.
# DataLoader формирует батчи примеров из dataset, которые передаются в нейронную сеть

train_params = {
    'batch_size': TRAIN_BATCH_SIZE,
    'shuffle': True,
    'num_workers': 0
}

val_params = {
    'batch_size': VALID_BATCH_SIZE,
    'shuffle': True,
    'num_workers': 0
}

training_loader = DataLoader(training_set, **train_params)
validation_loader = DataLoader(validation_set, **val_params)

Создадим модель на основе BERT с добавлением полносвязного слоя для решения задачи классификации.

In [17]:
NUM_CLASSES = 14

class BERTClass(torch.nn.Module):
  def __init__(self, num_classes):
    super(BERTClass, self).__init__()
    self.l1 = transformers.BertModel.from_pretrained('DeepPavlov/rubert-base-cased')
    self.l2 = torch.nn.Linear(768, num_classes)

  def forward(self, ids, mask, token_type_ids):
    x = self.l1(ids, attention_mask=mask, token_type_ids=token_type_ids)
    x = self.l2(x['pooler_output'])
    return x

# class BERTClass(torch.nn.Module):
#     def __init__(self, num_classes: int):
#         super(BERTClass, self).__init__()
#         self.bert = BertModel.from_pretrained('sberbank-ai/ruBERT-base')  # Используйте другую предобученную модель
#         self.fc = torch.nn.Linear(768, num_classes)  # Учитывая размер выходного вектора

#     def forward(self, ids, mask, token_type_ids):
#         x = self.bert(ids, attention_mask=mask, token_type_ids=token_type_ids)
#         x = self.fc(x['pooler_output'])
#         return x


  # def forward(self, ids, mask, token_type_ids):
  #   x = self.l1(ids, attention_mask=mask, token_type_ids=token_type_ids)
  #   x = self.l2(x['pooler_output'])
  #   return x


model = BERTClass(NUM_CLASSES)
model.to(device)

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BERTClass(
  (l1): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affi

В качестве функции потерь возьмем бинарную кросс-энтропию с сигмоидой, она используется в задачах классификации, где выходные данные модели - значения вероятности от 0 до 1.

В качестве оптимизатора возьмем SGD (стохастический градиентный спуск). Этот оптимизатор вычисляет градиент не по всей выборке, а по мини-батчу.

In [18]:
# def loss_fn(outputs, targets):
#   return torch.nn.BCEWithLogitsLoss()(outputs, targets)

# optimizer = torch.optim.SGD(params=model.parameters(), lr=LEARNING_RATE)

def loss_fn(outputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [19]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

Обучение и валидация:

In [20]:
def train(epoch):
    model.train()
    pbar = tqdm(training_loader)
    for _, data in enumerate(pbar, 0):
        ids = data["ids"].to(device, dtype=torch.long)
        mask = data["mask"].to(device, dtype=torch.long)
        token_type_ids = data["token_type_ids"].to(device, dtype=torch.long)
        targets = data["targets"].to(device, dtype=torch.float)

        optimizer.zero_grad() # обнуляем градиент
        outputs = model(ids, mask, token_type_ids)

        loss = loss_fn(outputs, targets)
        loss.backward()  # вычисляем обратное распространение ошибки
        optimizer.step() # обновляем веса модели

        if _ % 100 == 0:
          pbar.set_description(f"Epoch: {epoch + 1}, Batch: {_}, Loss: {loss.item()}")

In [21]:
def validation():
    model.eval()
    fin_targets = []
    fin_outputs = []

    with torch.no_grad():
        for _, data in enumerate(tqdm(validation_loader), 0):
            ids = data['ids'].to(device, dtype=torch.long)
            mask = data['mask'].to(device, dtype=torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)
            targets = data['targets'].to(device, dtype=torch.float)

            outputs = model(ids, mask, token_type_ids)

            # для вычисления метрик на валидации переносим данные на cpu и преобразуем к формату numpy
            fin_targets.extend(targets.cpu().detach().numpy())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy())

    # сеть выдала значения с плавающей точкой, преобразуем значения, большие threshold,
    # к 1, а меньшие threshold - к 0
    threshold = 0.5
    fin_outputs = (np.array(fin_outputs) >= threshold).astype(int)
    fin_targets = np.array(fin_targets)

    return fin_outputs, fin_targets

In [22]:
for epoch in range(EPOCHS):
    train(epoch)

    outputs, targets = validation()

    accuracy = metrics.accuracy_score(targets, outputs)
    f1_score_micro = metrics.f1_score(targets, outputs, average='micro', zero_division=0)
    f1_score_macro = metrics.f1_score(targets, outputs, average='macro', zero_division=0)

    print(f"Epoch {epoch+1} - Accuracy Score = {accuracy}")
    print(f"Epoch {epoch+1} - F1 Score (Micro) = {f1_score_micro}")
    print(f"Epoch {epoch+1} - F1 Score (Macro) = {f1_score_macro}")

100%|██████████| 8/8 [00:04<00:00,  1.77it/s]


Epoch 1 - Accuracy Score = 0.0
Epoch 1 - F1 Score (Micro) = 0.0
Epoch 1 - F1 Score (Macro) = 0.0


100%|██████████| 8/8 [00:05<00:00,  1.49it/s]


Epoch 2 - Accuracy Score = 0.008771929824561403
Epoch 2 - F1 Score (Micro) = 0.22346368715083798
Epoch 2 - F1 Score (Macro) = 0.08333333333333333


100%|██████████| 8/8 [00:04<00:00,  1.75it/s]


...

Epoch 8 - Accuracy Score = 0.12280701754385964
Epoch 8 - F1 Score (Micro) = 0.596774193548387
Epoch 8 - F1 Score (Macro) = 0.4480511652391739


100%|██████████| 8/8 [00:04<00:00,  1.74it/s]


Epoch 9 - Accuracy Score = 0.16666666666666666
Epoch 9 - F1 Score (Micro) = 0.6086956521739131
Epoch 9 - F1 Score (Macro) = 0.4539315519174401


100%|██████████| 8/8 [00:04<00:00,  1.72it/s]

Epoch 10 - Accuracy Score = 0.14912280701754385
Epoch 10 - F1 Score (Micro) = 0.6493506493506493
Epoch 10 - F1 Score (Macro) = 0.5352634083999411


Сохраним и скачаем веса модели

In [23]:
torch.save(model.state_dict(), 'model_weights.pth')

In [24]:
from google.colab import files
files.download('model_weights.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Предсказания на тестовом наборе

Установим и импортируем библиотеки:

In [ ]:
!pip install transformers

In [25]:
import pandas as pd
import numpy as np
import transformers
import torch

from sklearn import metrics
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel, BertConfig
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm import tqdm

In [26]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

Скачаем набор с публичными тестовыми данными (public_test.csv), прочитаем его.

Если указанная команда не заработала, скопируйте архивы с данными на свой Google Drive и укажите id архивов из ссылок к вашему Google Drive.

In [ ]:
!gdown 13c4PfMp-f4EKKdpSV6twqnYu7J-psYTm

Downloading...
From: https://drive.google.com/uc?id=13c4PfMp-f4EKKdpSV6twqnYu7J-psYTm
To: /content/public_test.csv
100% 923k/923k [00:00<00:00, 79.6MB/s]


In [27]:
df = pd.read_csv('private_test.csv')
df = df.rename({'Фильм': 'movie', 'Сюжет': 'plot', 'Описание': 'description'}, axis=1)
df

,movie,description,plot
0,Пираты галактики Барракуда (2024),На Землю попадает пришелец из далёкой галактик...,На Землю попадает пришелец из далёкой галактик...
1,Брат (1997),Демобилизованный из армии Данила Багров возвра...,Осень 1996 года. Вернувшийся с Чеченской войны...
2,Смерч 2 (2024),Кейт Картер изучает метеорологию в Нью-Йорке. ...,Кейт Картер изучает метеорологию в Нью-Йорке. ...
3,Реквием по мечте (2000),"В драме ""Реквием по мечте"" Сара Голдфарб мечта...",Главные герои: мать — Сара (Эллен Бёрстин) и с...
4,Капитан Волконогов бежал (2021),В Ленинграде 1938 года капитан Фёдор Волконого...,"Сюрреалистический Ленинград, 1938 год. Главный..."
...,...,...,...
118,Великий Гэтсби (2013),Весной 1922 года Ник Каррауэй приезжает в Нью-...,"Ник Кэррауэй, выпускник Йельского университета..."
119,Пришельцы (2022),В этом захватывающем блокбастере о путешествия...,NaN
120,Загадочная история Бенджамина Баттона (2008),"Фильм рассказывает историю Бенджамина Баттона,...",В августе 2005 года Дейзи Фуллер находится на ...
121,Отряд самоубийц (2016),После смерти Супермена правительство решает да...,После смерти Супермена агент разведки Аманда У...


In [ ]:
df.isnull().any()

Снова заполним пустые значения:

In [28]:
df['plot'].fillna(df['description'], inplace=True)
df.isnull().any()

,0
movie,False
description,False
plot,False


Создадим набор с тестовыми данными.

Создадим новый класс TestDataset, т.к. на этот раз в данных нет истинной разметки.

In [29]:
MAX_LEN = 512
tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')

class TestDataset(Dataset):

  def __init__(self, dataframe, tokenizer, max_len):
    self.tokenizer = tokenizer
    self.data = dataframe
    self.text = self.data['plot']
    self.max_len = max_len

  def __len__(self):
    return len(self.text)

  def __getitem__(self, index):
    text = str(self.text[index])

    inputs = self.tokenizer.encode_plus(
        text,
        None,
        add_special_tokens=True,
        max_length=self.max_len,
        padding='max_length',
        return_token_type_ids=True,
        truncation=True
    )
    ids = inputs["input_ids"]
    mask = inputs["attention_mask"]
    token_type_ids = inputs["token_type_ids"]

    return {
        'ids': torch.tensor(ids, dtype=torch.long),
        'mask': torch.tensor(mask, dtype=torch.long),
        'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long)
    }

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [30]:
public_set = TestDataset(df, tokenizer, MAX_LEN)
public_dataloader = DataLoader(public_set, batch_size=8, shuffle=False)

Создадим модель и загрузим веса

In [31]:
NUM_CLASSES = 14

class BERTClass(torch.nn.Module):
  def __init__(self, num_classes):
    super(BERTClass, self).__init__()
    self.l1 = transformers.BertModel.from_pretrained('DeepPavlov/rubert-base-cased')
    self.l2 = torch.nn.Linear(768, num_classes)

  def forward(self, ids, mask, token_type_ids):
    x = self.l1(ids, attention_mask=mask, token_type_ids=token_type_ids)
    x = self.l2(x['pooler_output'])
    return x
# class BERTClass(torch.nn.Module):
#     def __init__(self, num_classes: int):
#         super(BERTClass, self).__init__()
#         self.bert = BertModel.from_pretrained('sberbank-ai/ruBERT-base')  # Используйте другую предобученную модель
#         self.fc = torch.nn.Linear(768, num_classes)  # Учитывая размер выходного вектора

#     def forward(self, ids, mask, token_type_ids):
#         x = self.bert(ids, attention_mask=mask, token_type_ids=token_type_ids)
#         x = self.fc(x['pooler_output'])
#         return x


model = BERTClass(NUM_CLASSES)
model.to(device)

Some weights of the model checkpoint at DeepPavlov/rubert-base-cased were not used when initializing BertModel: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


BERTClass(
  (l1): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affi

In [32]:
path_to_weights = './model_weights.pth'
model.load_state_dict(torch.load(path_to_weights))

<ipython-input-32-d061cf6adefc>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(path_to_weights))


<All keys matched successfully>

Сделаем предсказания на тестовом наборе:

In [33]:
model.eval()
fin_outputs = []

with torch.no_grad():
    for data in tqdm(public_dataloader):
        ids = data['ids'].to(device, dtype=torch.long)
        mask = data['mask'].to(device, dtype=torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)

        outputs = model(ids, mask, token_type_ids)

        fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())

threshold = 0.5
fin_outputs = (np.array(fin_outputs) >= threshold).astype(int)

100%|██████████| 16/16 [00:04<00:00,  3.70it/s]


Возьмем полученный при обучении текстовый список жанров и преобразуем ответы сети к текстовому виду. Возьмем строковые значения жанров, соответствующие индексам единиц в массиве предсказаний, и объединим их в одну строку через запятую.

In [34]:
genres = ['боевик', 'детектив', 'драма', 'история', 'комедия', 'криминал',
       'мелодрама', 'мультфильм', 'мюзикл', 'приключения', 'триллер',
       'ужасы', 'фантастика', 'фэнтези']

In [35]:
def replace_and_join(array, genres):
    result = []
    for row in array:
        genre_indices = np.where(row == 1)[0]
        genre_names = [genres[i] for i in genre_indices]
        result.append(", ".join(genre_names))
    return result

answers = replace_and_join(fin_outputs, genres)

In [36]:
answers_df = df[['movie']].copy()
answers_df['genres'] = answers
answers_df = answers_df.rename({'movie': 'Фильм', 'genres': 'Жанры'}, axis=1)
answers_df

,Фильм,Жанры
0,Пираты галактики Барракуда (2024),"комедия, приключения"
1,Брат (1997),"боевик, комедия, криминал"
2,Смерч 2 (2024),драма
3,Реквием по мечте (2000),драма
4,Капитан Волконогов бежал (2021),"боевик, драма, история, криминал"
...,...,...
118,Великий Гэтсби (2013),драма
119,Пришельцы (2022),"боевик, приключения, фантастика"
120,Загадочная история Бенджамина Баттона (2008),"боевик, драма, история"
121,Отряд самоубийц (2016),"боевик, комедия"


Сохраним сделанные предсказания в .csv файл.

Добавим в название файла значение seed, сгенерированное при обучении сети. Это нужно для того, чтобы полученный в ходе обучения результат был повторим.

In [37]:
answers_df.to_csv(f'answers_seed_{seed}.csv', index=False)

In [38]:
from google.colab import files
files.download(f'answers_seed_{seed}.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>